# 3. Testing the agent loop without an LLM

**Time: about 15 minutes**

The loop in `run_agent` is our code, not the model's. It can break in boring ways:
a crash on a tool name the model made up, broken JSON, a loop that never ends.

We do not need a real model to test any of that. We need a **fake** one that says exactly what we tell it to.

## A fake model

`FakeLLM` has the same shape as the OpenAI client (`client.chat.completions.create(...)`).
It plays back a script of replies, one per call, and records every request the agent sent.

In [1]:
from IPython.display import Code

Code(filename="tests/fake_llm.py", language="python")

"""A fake OpenAI client for tests. It plays back a script of replies, so the agent loop runs with no API key."""

import json
from types import SimpleNamespace


def tool_call(name: str, call_id: str = "call_1", **args) -> SimpleNamespace:
    """One tool call, shaped like the ones the OpenAI API returns."""
    return SimpleNamespace(id=call_id, type="function", function=SimpleNamespace(name=name, arguments=json.dumps(args)))


def reply(content: str | None = None, tool_calls: list | None = None) -> SimpleNamespace:
    """One response from the model: either some text, or a list of tool calls."""
    message = SimpleNamespace(role="assistant", content=content, tool_calls=tool_calls)
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])


class FakeLLM:
    """Use it like OpenAI(): fake.chat.completions.create(...). Every call returns the next scripted reply."""

    def __init__(self, replies: list):
        self.replies = list(replies)
        self.requests = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, **request):
        self.requests.append({**request, "messages": list(request["messages"])})
        if not self.replies:
            raise AssertionError("The agent called the model more times than the script expected")
        return self.replies.pop(0)

This only works because `run_agent` accepts a `client`. That one parameter is what makes the agent testable.
This idea is called **dependency injection**: the code receives what it depends on, instead of creating it inside.

In [2]:
from agent import run_agent
from tests.fake_llm import FakeLLM, reply, tool_call

fake = FakeLLM([reply("Hi! What would you like to eat?")])
run_agent("Hello", client=fake)

{'answer': 'Hi! What would you like to eat?', 'tools_called': []}

## A script with a tool call

Step 1: the "model" asks for `track_order`. Step 2: it answers. Our loop has to run the real tool in between,
and send the result back.

In [3]:
fake = FakeLLM([
    reply(tool_calls=[tool_call("track_order", order_id="QB-1001")]),
    reply("Your order is out for delivery."),
])
result = run_agent("Where is QB-1001?", client=fake)

print("Answer:", result["answer"])
print("Tools:", result["tools_called"])
print("What the model saw on its second turn:", fake.requests[1]["messages"][-1])

Answer: Your order is out for delivery.
Tools: [{'name': 'track_order', 'args': {'order_id': 'QB-1001'}, 'output': '{"order_id": "QB-1001", "status": "Out for delivery, arriving in 10 minutes"}'}]
What the model saw on its second turn: {'role': 'tool', 'tool_call_id': 'call_1', 'content': '{"order_id": "QB-1001", "status": "Out for delivery, arriving in 10 minutes"}'}


## The cases that break in production

Real models sometimes call tools that do not exist, pass the wrong arguments, or send broken JSON.
Our loop should turn each of these into an error message the model can read and recover from, never a crash.

In [4]:
scripts = {
    "unknown tool": tool_call("delete_database"),
    "wrong argument name": tool_call("get_menu", shop="Green Bowl"),
}
broken_json = tool_call("get_menu")
broken_json.function.arguments = '{"restaurant": "Green Bo'
scripts["broken JSON"] = broken_json

for label, bad_call in scripts.items():
    fake = FakeLLM([reply(tool_calls=[bad_call]), reply("Sorry, let me try again.")])
    result = run_agent("Menu please", client=fake)
    print(f"{label:20} -> {result['tools_called'][0]['output']}")

unknown tool         -> {"error": "Unknown tool: delete_database"}
wrong argument name  -> {"error": "Bad arguments for get_menu: get_menu() got an unexpected keyword argument 'shop'"}
broken JSON          -> {"error": "Arguments for get_menu were not valid JSON"}


And a model that never stops calling tools. Without `max_steps` this would loop, and bill you, forever:

In [5]:
fake = FakeLLM([reply(tool_calls=[tool_call("track_order", order_id="QB-1001")])] * 3)
result = run_agent("Where is my order?", client=fake, max_steps=3)
print("Model calls:", len(fake.requests))
print("Answer:", result["answer"])

Model calls: 3
Answer: Sorry, I could not finish that request. Please try again.


## The tests

Each demo above is now a test in `tests/test_agent_loop.py`:

In [6]:
!pytest tests/test_agent_loop.py -v

============================= test session starts ==============================
platform darwin -- Python 3.12.16, pytest-9.2.0, pluggy-1.7.0 -- /Users/datasense/Desktop/agent-evals-ci/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/datasense/Desktop/agent-evals-ci
configfile: pytest.ini
plugins: platformdirs-4.14.5, deepeval-4.3.2, repeat-0.9.4, xdist-3.8.0, anyio-4.16.0, rerunfailures-16.7, asyncio-1.4.1.post1
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=function, asyncio_default_test_loop_scope=function
collected 8 items                                                              

tests/test_agent_loop.py::test_returns_the_answer_when_no_tool_is_needed PASSED [ 12%]
tests/test_agent_loop.py::test_sends_the_system_prompt_the_tools_and_temperature_zero PASSED [ 25%]
tests/test_agent_loop.py::test_runs_a_tool_and_sends_the_result_back_to_the_model PASSED [ 37%]
tests/test_agent_loop.py::test_runs_several_tool_calls_from_one_reply PASSED [ 50%]


## Everything so far

All three test files together. Note the time: this is what runs on every push before we spend a cent on evals.

In [7]:
!pytest tests -q

..................................                                       [100%]Running teardown with pytest sessionfinish...

34 passed in 0.02s


## What these tests cannot tell you

They prove our code works **if** the model behaves a certain way. They say nothing about whether the real model
picks the right tools, passes the right arguments, or gives a good answer. For that we need the real model,
and a way to score fuzzy output. That is what evals are for, in the next notebook.

## Try it yourself

1. Write a test where the model calls `place_order` for a closed restaurant. Check that the error reaches the model.
2. Write a test for an empty reply (`reply(None)`). What should the agent return? Is the current behavior right?
3. Python's `unittest.mock.MagicMock` can also fake the client. Rewrite one test with it. Which version is easier to read?